**Introduction**
The motivation behind this project was to learn more about the quick application of ML algorithms by experimenting with various classification models and the tuning parameters associated with each model. 
Caret is a powerful R library which is used for Classifaction And REgression Training, and I wanted to get familiar with this package and different classification algorithms. 

**Data**
This dataset consists of the: 
* Refractive Index 
* Measures of various elements in the glass
* The Glass Type (1-7) eg. *for headlamps, containers, building windows, vehicle windows etc. *** (Target Variable)**

**Aim**
* The aim was to identify patterns between variables in the dataset through Principle Component Analysis (PCA) 
* Apply and evaluate different classification models and see how each one performs based on the 'Accuracy' and 'Kappa' metrics 

*I expected the Random Forest model to perform the best, so just felt lke exploring different parameter tuning methods (randomSearch and gridSearch through caret)*

In [ ]:
suppressMessages(library(tidyverse))
suppressMessages(library(corrplot))
suppressMessages(library(ggfortify))
suppressMessages(library(e1071))
suppressMessages(library(caret))
suppressMessages(library(glmnet))


In [ ]:
glass <- read.csv('../input/glass.csv')

head(glass)

In [ ]:
colSums(is.na(glass))

**Exploratory Data Analysis**

In [ ]:
ggplot(glass, aes(Type, fill = Type)) + 
  geom_histogram() + 
  labs(title = "Glass Type count") + 
  scale_x_continuous(breaks = seq(1,7,1)) +
  scale_y_continuous(breaks = seq(0,100,10))

In [ ]:
glass$Type <- as.factor(glass$Type)
ggplot(glass, aes(x = Type, colour = Type)) + 
  geom_density(aes(group = Type, fill = Type), alpha = 0.3) +
  labs(title = "Distribution of each Type")

**Correlations Heatmap**

In [ ]:
corrGlass <- round(cor(glass[1:9]), 3)
corrplot(corrGlass, method = "color", tl.col = "black", tl.cex = 0.65)

**Principle Component Analysis**

Principle Component Analysis (PCA) is a dimensions reduction technique that uses linear algebra. This technique is often used in large datasets. 
This technique transforms correlated variables into smaller uncorrelated variables which are called 'Principle Components'. Each of these components accounts for a certain percentage variability in the data. The first principle component explains the most variation followed the second and so on. 
Often each principle components display an interpretable pattern of which we can draw relationship based insight from. 

In [ ]:
pcaGlass <- prcomp(glass[,1:9])
plot(pcaGlass, type = "l")

summary(pcaGlass)

autoplot(prcomp(glass[,1:9]), data = glass, colour = 'Type', 
         frame = T, frame.type = 'norm') +
  geom_vline(xintercept=c(-0,0), linetype="dashed", size=0.3) + 
  geom_hline(yintercept=c(-0,0), linetype="dashed", size=0.3)

PC1 explains 47.62% of the variation in the data, and PC2 explains 26.32%. 
Quite to difficult to interpret components based on 'Type' clusters since they are mostly stacked on each others.

**Prepare Data for Modelling**

In [ ]:
inTrain <- createDataPartition(y = glass$Type, p = 0.7, list = F)
trainingGlass <- glass[inTrain,]
testingGlass <- glass[-inTrain,]

**Multinomial Logistic Regression**
This is a classifaction technique quite similar to logistic regression however the dependent (target) variable consists of more than two level. It is used to describe the data and to expalin the relationship between one dependent nominal variable and multiple independent variables. 
In our case, we will attempt to explain the relationship between Glass Type and independent variables such as Refractive Index and elements quantities. 

In [ ]:
# Multinomial Logistic Regression 
mlrcontrol <- trainControl(method = "cv", number = 10, repeats = 2)
mlr <- train(Type~., trainingGlass, method = "glmnet", 
             metric = "Accuracy", 
             tuneGrid = expand.grid(alpha = 0.1, lambda = 0.004972569), 
             trControl = mlrcontrol)

mlrPredictGlass <- predict(mlr, testingGlass)
confusionMatrix(testingGlass$Type, mlrPredictGlass)

**Support Vector Machine**
The SVM algorithm is implemented using a kernel. We plot the data as a point in a mutli-dimensional space with the value of each feature being the value of a particular co-ordinate. The algorithm outputs an optimal hyperplane which categorizes new data samples. The optimal hyperplane is found through support vectors, which are co-ordinates of individual observations and it's a frontier which best segregates the classes. 

In [ ]:
# SVM 
svmControl <- trainControl(method = "cv", number = 10, repeats = 2)
svmModelGlass <- train(Type~., trainingGlass, 
                       method = "svmRadial", 
                       preProcess = c("center","scale"),
                       metric = "Accuracy",
                       tuneGrid = expand.grid(.sigma = 0.5, .C = 10), 
                       trControl = svmControl)

svmPredictGlass <- predict(svmModelGlass, testingGlass)
confusionMatrix(testingGlass$Type, svmPredictGlass)

**K-Nearest Neighbours** This is non-parameric method used for classifcation, where data points are separated in several classes and the input consists of the k-closest training examples in the feature space

In [ ]:
# K-Nearest Neighbours
knnModelGlass <- train(Type~., trainingGlass, 
                       method = "knn", 
                       preProcess = c("center","scale"), 
                       tuneLength = 10, 
                       trControl = trainControl(method = "cv", number = 10, repeats = 2))

knnPredictGlass <- predict(knnModelGlass, testingGlass)
confusionMatrix(knnPredictGlass, testingGlass$Type)

**Random Forest** These are an ensemble learning method for classification and regression, that operate by constructing a multitude of decision trees at training time and outputing the class that is the mode of the classes in classification. 

In [ ]:
# Random Forest 
rf.grid <- expand.grid(.mtry = 2:9)
rf <- train(Type~., data = trainingGlass, 
            method = 'rf',
            metric = "Accuracy",
            ntree = 500, 
            tuneGrid = rf.grid, 
            trControl = trainControl(method = "cv", number = 10, repeats = 2))
print(rf)

In [ ]:
# Random search Tuning 
controlRandom <- trainControl(method = "cv", number = 10, repeats = 2, search = "random")
metric <- "Accuracy"
set.seed(320)

rfGlassRandom <- train(Type~., trainingGlass,
                 method = "rf", 
                 preProcess = c("center","scale"), 
                 metric = metric, 
                 tuneLength = 15, 
                 trControl = controlRandom)
print(rfGlassRandom)

In [ ]:
# Grid search tuning 
controlGrid <- trainControl(method = "cv", number = 10, repeats = 2, search = "grid")
set.seed(321)

rfGlassGrid <- train(Type~., trainingGlass,
                 method = "rf", 
                 preProcess = c("center","scale"), 
                 metric = metric, 
                 tuneGrid = rf.grid, 
                 trControl = controlGrid)
print(rfGlassGrid)

In [ ]:
ggplot(rfGlassRandom) +
  geom_line()
ggplot(rfGlassGrid)

In [ ]:
rfPredictGlass <- predict(rf, testingGlass)
confusionMatrix(rfPredictGlass,testingGlass$Type)

**Neural Network** Neural networks consist of inut and output layers, as well as a hidden layer consisting of units that transform the input into something the output layer can use. 

In [ ]:
# Neural Network 
nnetcontrol <- trainControl(method = "cv", number = 10, repeats = 2) 
nnetModelGlass <- train(Type~., trainingGlass, 
                        method = 'nnet',
                        metric = "Accuracy",
                        preProcess = c("center","scale"),
                        tuneGrid = expand.grid(size = 10, decay = 0.1), 
                        verbose = F, 
                        trControl = nnetcontrol)

nnetPredictGlass <- predict(nnetModelGlass, testingGlass)
confusionMatrix(nnetPredictGlass, testingGlass$Type)

In [ ]:
modelResamples <- resamples(list(MLR = mlr, SVM = svmModelGlass, KNN = knnModelGlass, RF = rf, NNet = nnetModelGlass))

densityplot(modelResamples, metric = "Kappa", auto.key = list(coloumns = 3))

In [ ]:
bwplot(modelResamples, metric = c("Kappa", "Accuracy"))

**Conclusions** As expected, Random Forest out performed all other models based on our metrics. 